# Дообучение YandexGPT-5-Lite-8B-pretrain

Ноутбук подготавливает LoRA/SFT-обучение для задачи преобразования русских аналитических запросов в JSON-фильтры API.


## Что нужно до запуска

- GPU с CUDA и желательно 16+ GB VRAM но у меня 8 GB и работает
- `HF_TOKEN` в корневом `.env`
- датасет уже лежит в `training/llm_sft/`


In [1]:
!pip install -q datasets transformers accelerate peft trl bitsandbytes sentencepiece huggingface_hub


In [2]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import torch
from huggingface_hub import login
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


/media/timur/G/PycharmProjects/amur-hakaton-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_env_file(repo_root / '.env')
hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN не найден в .env')

login(token=hf_token, add_to_git_credential=False)
print('HF token loaded successfully')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF token loaded successfully


In [4]:
MODEL_NAME = 'yandex/YandexGPT-5-Lite-8B-pretrain'
TRAIN_PATH = repo_root / 'training' / 'llm_sft' / 'budget_query_sft_train.jsonl'
VAL_PATH = repo_root / 'training' / 'llm_sft' / 'budget_query_sft_val.jsonl'
DATASET_CACHE_DIR = repo_root / '.cache' / 'hf_datasets'
OUTPUT_DIR = repo_root / 'artifacts' / 'yandexgpt5-lite-budget-query-lora'

DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MAX_SEQ_LENGTH = 1536
print(TRAIN_PATH)
print(VAL_PATH)
print(DATASET_CACHE_DIR)


/media/timur/G/PycharmProjects/amur-hakaton-2026/training/llm_sft/budget_query_sft_train.jsonl
/media/timur/G/PycharmProjects/amur-hakaton-2026/training/llm_sft/budget_query_sft_val.jsonl
/media/timur/G/PycharmProjects/amur-hakaton-2026/.cache/hf_datasets


In [5]:
# Explicit features avoid the null/string inference issue in datasets.load_dataset('json', ...).
from training.llm_sft.dataset_loader import load_budget_query_sft_dataset

dataset = load_budget_query_sft_dataset(cache_dir=DATASET_CACHE_DIR)
dataset


DatasetDict({
    train: Dataset({
        features: ['task', 'text_query', 'prompt', 'completion', 'target'],
        num_rows: 426
    })
    validation: Dataset({
        features: ['task', 'text_query', 'prompt', 'completion', 'target'],
        num_rows: 80
    })
})

In [6]:
sample = dataset['train'][0]
print(sample['text_query'])
print(sample['prompt'][:700])
print(sample['completion'])


Покажи объем соглашений по Свободному по месяцам
Ты преобразуешь русский запрос к системе бюджетной аналитики в JSON для API.
Верни только JSON без пояснений.

Требуемая схема:
- date_from: строка в формате YYYY-MM-DD или null
- date_to: строка в формате YYYY-MM-DD или null
- metrics: массив строк или null
- filters: объект со следующими ключами:
  source_groups, object_query, budget_query, organization_query, document_id,
  document_number, kfsr_code, kcsr_code, kvr_code, kvsr_code, kesr_code,
  kosgu_code, purpose_code, funding_source
- group_by: массив строк или null

Правила:
- если пользователь просит один показатель, верни ровно один metric
- если пользователь не указал даты, верни null
- если пользователь просит по месяцам, верни gr
{
  "date_from": null,
  "date_to": null,
  "metrics": [
    "agreement_amount"
  ],
  "filters": {
    "source_groups": null,
    "object_query": "Свободный",
    "budget_query": null,
    "organization_query": null,
    "document_id": null,
    "do

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Для дообучения нужен GPU с CUDA. На CPU этот ноутбук не рассчитан.')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token, legacy=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    device_map="auto",
    dtype=torch.float16,
    quantization_config=bnb_config,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    completion_only_loss=True,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()


In [ ]:
trainer.train()


In [ ]:
trainer.save_model(str(OUTPUT_DIR / 'adapter'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'adapter'))


In [ ]:
test_queries = [
    'Покажи лимиты по Благовещенску по месяцам',
    'Покажи кассовые выплаты по 0502',
    'Покажи сумму контрактов по источнику gz',
]

def build_inference_prompt(query: str) -> str:
    return (
        'Ты преобразуешь русский запрос к системе бюджетной аналитики в JSON для API. '
        'Верни только JSON без пояснений.\n\n'
        f'Запрос пользователя:\n{query}\n\nВерни только JSON:'
    )

for query in test_queries:
    prompt = build_inference_prompt(query)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=220, temperature=0.0, do_sample=False)
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('QUERY:', query)
    print(generated)
    print('-' * 80)
